# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karthikj5453/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane 2: Refresh / Content Opportunity Scoring** (lane confirmed, not switching).
**Data:** the bundled starter slice `data/raw/content_refresh_anonymized.csv` (30,000 pages, 32 clients). Rows are trailing-90-day totals, so there is no time axis to leak across; `trend_direction` is used **only** to grade the queue afterwards, never as an input.

In [1]:
import json
from pathlib import Path
import numpy as np, pandas as pd

ROOT = Path.cwd()
while not (ROOT / "scripts").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
OUT = ROOT / "work" / "outputs"; OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(ROOT / "data/raw/content_refresh_anonymized.csv")
# Grading label ONLY (never an input to the rule): last-30d impressions fell >20% vs prior 30d.
df["declining"] = (df["trend_direction"] == "down").astype(int)
BASE = df["declining"].mean()
print(f"rows={len(df):,}  clients={df['client_id'].nunique()}  base rate of declining={BASE:.3f}")

rows=30,000  clients=32  base rate of declining=0.542


## 1) The rule in plain words, then two signal checks

**Rule:** a page is worth a refresh review if it **has not been updated for a while** *and* it **still gets real search demand**; rank by demand (capped, so a few giant pages don't swamp the list) times how long it has been stale.

That rule leans on two signals. Check both before encoding anything. Tables show **n** and the share of pages that were declining (the grading label), with lift vs the base rate.

### Signal A — staleness (the signal behind FlyRank's refresh flags)

In [2]:
def bucket_table(col, order):
    t = (df.groupby(col)["declining"].agg(n="size", declining_rate="mean")
           .reindex(order))
    t["lift_vs_base"] = t["declining_rate"] / BASE
    return t.round(3)

sig_a = bucket_table("freshness_tier", ["0-30", "31-90", "91-180", "181+"])
print(f"base rate = {BASE:.3f}")
sig_a

base rate = 0.542


,n,declining_rate,lift_vs_base
freshness_tier,,,
0-30,20480,0.511,0.943
31-90,175,0.589,1.086
91-180,9171,0.611,1.127
181+,174,0.471,0.869


**Verdict A: MIXED.** Pages last updated 91–180 days ago decline more often than fresh ones (0–30 days), and that tier has thousands of pages, so the direction holds where n is large. But it is **not monotonic**: the 181+ tier declines *less* than average, on only n=174 pages, and 31–90 has n=175. So "older means more declining" is true in the big middle and unproven at the extremes. I encode staleness as a **91+ days threshold**, not as "the older the better without limit" — and I flag in the review that very old pages may be deliberately left alone.

### Signal B — search volume / visibility (the signal behind quick-win logic)

In [3]:
sig_b = bucket_table("impression_tier", ["low", "moderate", "good", "excellent"])
print(f"base rate = {BASE:.3f}")
sig_b

base rate = 0.542


,n,declining_rate,lift_vs_base
impression_tier,,,
low,11248,0.454,0.837
moderate,10469,0.615,1.134
good,7205,0.586,1.081
excellent,1078,0.462,0.852


**Verdict B: MIXED (inverted-U).** Demand matters up to a point: `moderate` and `good` pages decline more than `low`, but `excellent` pages (30,000+ impressions) decline *less* than average. A rule that simply multiplies by raw impressions would push the biggest, steadiest pages to the top of the queue. This negative result changes the rule: **volume is capped at 3,000 impressions** before it enters the score, and a floor of 300 impressions (the `moderate` tier boundary) filters out low-signal pages.

## 2) Encode ONE rule: score, one reason code, one action label

```
eligible = days_since_last_update >= 91  AND  impressions_90d >= 300
score    = min(impressions_90d, 3000) * days_since_last_update      (0 if not eligible)
reason   = "stale_with_demand"
action   = "refresh"
```
Inputs are only `days_since_last_update` and `impressions_90d`. No `trend_*` column, no label, no ID.

In [4]:
STALE_DAYS, MIN_IMPR, CAP = 91, 300, 3000

eligible = (df["days_since_last_update"] >= STALE_DAYS) & (df["impressions_90d"] >= MIN_IMPR)
df["score"] = np.where(eligible,
                       np.minimum(df["impressions_90d"], CAP) * df["days_since_last_update"], 0)
df["reason_code"] = np.where(eligible, "stale_with_demand", "not_flagged")
df["action"] = np.where(eligible, "refresh", "monitor")

queue = (df[df["score"] > 0]
         .sort_values("score", ascending=False)
         .reset_index(drop=True))
queue["rank"] = queue.index + 1

cols = ["rank", "content_id", "client_id", "score", "reason_code", "action",
        "impressions_90d", "days_since_last_update", "avg_position", "ctr", "content_type"]
queue[cols].to_csv(OUT / "baseline_action_score.csv", index=False)
print(f"flagged {len(queue):,} of {len(df):,} pages -> work/outputs/baseline_action_score.csv")
queue[cols].head(10)

flagged 7,234 of 30,000 pages -> work/outputs/baseline_action_score.csv


,rank,content_id,client_id,score,reason_code,action,impressions_90d,days_since_last_update,avg_position,ctr,content_type
0,1,content_fe16a55cd13d,client_7f2253d7e2,582000,stale_with_demand,refresh,4556,194,16.4,0.33,keyword article
1,2,content_cf56e2e2e282,client_7f2253d7e2,582000,stale_with_demand,refresh,61678,194,19.7,0.15,keyword article
2,3,content_b16bd7307b39,client_7f2253d7e2,582000,stale_with_demand,refresh,4590,194,31.0,0.00,keyword article
3,4,content_1bfaa38ff26c,client_7f2253d7e2,582000,stale_with_demand,refresh,25715,194,22.2,0.23,keyword article
4,5,content_5feee3994adb,client_7f2253d7e2,582000,stale_with_demand,refresh,7812,194,39.0,0.01,keyword article
5,6,content_7368877ea310,client_7f2253d7e2,582000,stale_with_demand,refresh,59472,194,24.8,0.13,keyword article
6,7,content_ecb6215e79fd,client_7f2253d7e2,582000,stale_with_demand,refresh,4429,194,25.3,0.38,keyword article
7,8,content_c2d929d83eaa,client_7f2253d7e2,579000,stale_with_demand,refresh,7558,193,17.9,0.20,keyword article
8,9,content_0a91db491d14,client_7f2253d7e2,579000,stale_with_demand,refresh,13299,193,10.5,0.49,keyword article
9,10,content_cb7e312f5d32,client_9f14025af0,453000,stale_with_demand,refresh,21272,151,12.6,2.45,keyword article


In [5]:
def precision_at_k(k): return queue["declining"].head(k).mean()

metrics = {
    "n_rows": int(len(df)), "n_flagged": int(len(queue)),
    "base_rate_declining": round(float(BASE), 4),
    "flagged_rate_declining": round(float(queue["declining"].mean()), 4),
    "precision_at_10": round(float(precision_at_k(10)), 3),
    "precision_at_20": round(float(precision_at_k(20)), 3),
    "precision_at_50": round(float(precision_at_k(50)), 3),
    "precision_at_100": round(float(precision_at_k(100)), 3),
    "rule": {"stale_days": STALE_DAYS, "min_impressions": MIN_IMPR, "impression_cap": CAP,
             "score": "min(impressions_90d, cap) * days_since_last_update"},
}
json.dump(metrics, open(OUT / "baseline_metrics.json", "w"), indent=2)
pd.Series({k: v for k, v in metrics.items() if k != "rule"})

n_rows                    30000.0000
n_flagged                  7234.0000
base_rate_declining           0.5421
flagged_rate_declining        0.6175
precision_at_10               0.9000
precision_at_20               0.9000
precision_at_50               0.8800
precision_at_100              0.7500
dtype: float64

### Skeptic check — is one client doing the work?
The top of the queue looks suspiciously uniform. Count how many of the top-K pages come from a single client.

In [6]:
rows = []
for k in (10, 20, 50, 100):
    vc = queue.head(k)["client_id"].value_counts()
    rows.append({"top_k": k, "biggest_client_share": round(vc.iloc[0] / k, 2),
                 "n_clients_in_top_k": int(vc.size)})
conc = pd.DataFrame(rows)
lead = queue.head(50)["client_id"].value_counts().index[0]
lead_rate = df.loc[df["client_id"] == lead, "declining"].mean()
lead_days = df.loc[df["client_id"] == lead, "days_since_last_update"].value_counts().head(3).to_dict()
print(f"client leading the top 50: {lead}  | its overall declining rate = {lead_rate:.3f} "
      f"(n={int((df['client_id']==lead).sum())})  | most common days_since_last_update: {lead_days}")

# same rule, but at most 3 pages per client in the ranked list
div = queue[queue.groupby("client_id").cumcount() < 3]
metrics["top_client_share_top50"] = float(conc.loc[conc.top_k == 50, "biggest_client_share"].iloc[0])
metrics["lead_client_declining_rate"] = round(float(lead_rate), 3)
metrics["diversified_precision_at_10"] = round(float(div["declining"].head(10).mean()), 3)
metrics["diversified_precision_at_20"] = round(float(div["declining"].head(20).mean()), 3)
metrics["diversified_precision_at_50"] = round(float(div["declining"].head(50).mean()), 3)
json.dump(metrics, open(OUT / "baseline_metrics.json", "w"), indent=2)
display(conc)
print({k: v for k, v in metrics.items() if k.startswith("diversified")})

client leading the top 50: client_7f2253d7e2  | its overall declining rate = 0.937 (n=1043)  | most common days_since_last_update: {20: 976, 106: 53, 194: 9}


,top_k,biggest_client_share,n_clients_in_top_k
0,10,0.90,2
1,20,0.80,3
2,50,0.70,7
3,100,0.35,9


{'diversified_precision_at_10': 0.7, 'diversified_precision_at_20': 0.65, 'diversified_precision_at_50': 0.56}


**Reading the numbers.** The headline precision@K above is **inflated by one client**: a single client supplies most of the top 50, that client's pages are declining far more often than the base rate, and its nine stalest pages all show the identical `days_since_last_update` of 194 (a bulk-edit artifact is plausible), so "most stale" partly means "belongs to that client". With at most 3 pages per client, precision@50 falls to roughly the base rate. The diversified numbers (max 3 pages per client) are the fairer baseline. That fairer number, not the headline, is what the Week-5 model has to beat, so the rule is **frozen from here**.

*Honesty note:* I compared three score shapes (raw impressions, log impressions, capped impressions × days) and kept the one that follows from the two signal verdicts. That is mild tuning against the same labels used for grading, so all of these are optimistic. Week 5 must use a client-holdout split.

## 3) Top-10 review — action, why it's there, what would make it wrong

In [7]:
def wrong_if(r):
    pos = r["avg_position"]
    if pos == 0:
        a = "there is no position data, so 'demand' rests on impressions alone"
    elif pos <= 10:
        a = "it already ranks on page 1 and may simply be at its ceiling, so a refresh adds little"
    elif pos <= 20:
        a = "the 90-day total was one spike and demand is already gone"
    else:
        a = "a position past 20 means the demand is not realistically reachable"
    if r["days_since_last_update"] >= 181:
        b = "a 181+ day gap may mean the page is deliberately evergreen"
    else:
        b = "the last update was a trivial edit, so 'stale' says little about content quality"
    return f"wrong if {a}, or if {b}."

top10 = queue.head(10).copy()
for _, r in top10.iterrows():
    print(f"#{int(r['rank'])}  [{r['action']}]  {r['content_id']}\n"
          f"    why: reason={r['reason_code']}; {int(r['impressions_90d']):,} impressions (90d), "
          f"{int(r['days_since_last_update'])} days since update, position {r['avg_position']}\n"
          f"    {wrong_if(r)}\n")

#1  [refresh]  content_fe16a55cd13d
    why: reason=stale_with_demand; 4,556 impressions (90d), 194 days since update, position 16.4
    wrong if the 90-day total was one spike and demand is already gone, or if a 181+ day gap may mean the page is deliberately evergreen.

#2  [refresh]  content_cf56e2e2e282
    why: reason=stale_with_demand; 61,678 impressions (90d), 194 days since update, position 19.7
    wrong if the 90-day total was one spike and demand is already gone, or if a 181+ day gap may mean the page is deliberately evergreen.

#3  [refresh]  content_b16bd7307b39
    why: reason=stale_with_demand; 4,590 impressions (90d), 194 days since update, position 31.0
    wrong if a position past 20 means the demand is not realistically reachable, or if a 181+ day gap may mean the page is deliberately evergreen.

#4  [refresh]  content_1bfaa38ff26c
    why: reason=stale_with_demand; 25,715 impressions (90d), 194 days since update, position 22.2
    wrong if a position past 20 means th

## 4) Weak picks
Grade the top 20 against what actually happened (label used **only here**, after the fact).

In [8]:
top20 = queue.head(20)
weak = top20[top20["declining"] == 0]
print(f"top-20 weak picks (not declining): {len(weak)} of 20")
weak[["rank", "content_id", "impressions_90d", "days_since_last_update", "avg_position",
      "impressions_last_30d", "impressions_prev_30d", "trend_direction"]]

top-20 weak picks (not declining): 2 of 20


,rank,content_id,impressions_90d,days_since_last_update,avg_position,impressions_last_30d,impressions_prev_30d,trend_direction
9,10,content_cb7e312f5d32,21272,151,12.6,6881,3897,up
15,16,content_a5dbb404bdc2,79035,106,8.7,31757,30774,stable


**Named weakness:** the top of the queue is dominated by one client, so the rule partly ranks *clients*, not pages (see the skeptic check). Beyond that, the rule only sees *staleness × demand*. It cannot tell a page that is quietly growing or holding steady from one that is falling, because I deliberately kept every trend column out of the rule. The weak picks above are pages that are old and visible but not declining — exactly the false positives this baseline should be expected to make, and exactly what a model with legitimate signal has room to improve on.

## 5) Self-check
- [x] Two signals checked with bucket tables and n printed; at least one is flag-linked (staleness)
- [x] One-word verdict each (MIXED, MIXED) with the reason written out
- [x] One rule: a score, one reason code (`stale_with_demand`), an action label (`refresh`)
- [x] Ranked queue written by this notebook to `work/outputs/baseline_action_score.csv` (kept out of git by design)
- [x] Ten reviewed rows with "what would make it wrong"
- [x] No future-window or label-derived inputs to the rule (`trend_*` and `declining` are grading-only)
- [x] Skeptic check: client concentration in the top-K reported, with a diversified precision@K
- [x] Metrics receipt written to `work/outputs/baseline_metrics.json` (commit this)
- [ ] Executed notebook committed under `work/notebooks/`, then submit the repo URL